In [25]:
import matplotlib.pyplot as plt
import pandas as pd

from data_pipeline.loader import FluDataLoader
from data_pipeline.utils import get_holidays
from timeseriesutils import featurize

from datetime import timedelta
import polars as pl
import plotly.express as px

from jacques_flu.features import create_features_and_targets, train_test_split, split_features_labels

from jacques_flu.config import PROCESSED_DATA_DIR

In [26]:
flu_df = pl.read_csv(PROCESSED_DATA_DIR / "flu_features.csv", schema_overrides={'location': pl.String, 'wk_end_date': pl.Date})


## Flusight Targets
-   All 50 states, DC and PR
-   Primary:
    -    Quantile predictions for weekly laboratory confirmed influenza hospital admissions
-   Secondary
    -   Category probabilty predictions for direction and magnitude of changes in hospitalization per 100k rate
    -   Probability predictions for Peak week
    -   Quantile Predictions for peak incidence of hospital admisisons
    

# Let's visualize the data for a couple of states

## Massachusetts

In [ ]:
def plot_time_series(df, target, max_date, plot_weeks, time_var, group_var, group_value):
    """
    Plot a time series of the target variable for a given group.

    Parameters
    ---------- 
    df : pd.DataFrame
        Data to plot.
    target : str
        Name of the target variable.
    max_date : datetime
        Latest date to plot.
    plot_weeks : int
        Number of weeks to plot.
    time_var : str  
        Name of the time variable.
    group_var : str 
        Name of the group variable.
    group_value : str
        Value of the group variable to plot.
    """

    min_date = max_date - timedelta(weeks=plot_weeks)

    # Filter the data for the given state and date range
    state_data = (df.filter(pl.col('source') == 'HHS')
                  .filter(pl.col(group_var) == group_value)
                  .filter(pl.col(time_var) >= min_date)
                  .filter(pl.col(time_var) <= max_date)
                  .select([time_var, target]))
    # Create a time series plot using Plotly
    fig = px.line(state_data, x=time_var, y=target, title=f'Flu Counts in {group_value} from {min_date} to {max_date}')
    
    # Show the plot
    fig.show()

In [12]:
#Make plotting Data

#Limit to just HHS data, and one state
max_date = pd.to_datetime('2023-12-01')
plot_weeks = 100
time_var = 'wk_end_date'
group_var = 'location'
group_value = '01'
target = 'inc_trans'

In [13]:
min_date = max_date - timedelta(weeks=plot_weeks)

In [19]:
#Source options: 'hhs', 'flusurvnet', 'ilinet'

state_data = (flu_df.filter(pl.col('source') == 'hhs')
                .filter(pl.col(group_var) == group_value)
                .filter(pl.col(time_var) >= min_date)
                .filter(pl.col(time_var) <= max_date)
                .select([time_var, target])
                .sort(pl.col(time_var)))

In [23]:
fig = px.line(state_data, x=time_var, y=target, title=f'Flu Incidence in {group_value} from {min_date} to {max_date}')

fig.show()

In [ ]:
state_data

In [21]:
plot_time_series(flu_df, 'inc_trans', pd.to_datetime('2024-12-01'), 52, 'wk_end_date', 'location', '01')